# 00 · Colab setup and archive inventory

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya

**Priority:** 0 (Checkpoint 1)

**Purpose.** Verify the four OULU-NPU archives exist, record their sizes, and discover the internal layout (videos, annotations, protocol files, naming) by reading archive headers only.

**In plain words.** Before touching any video we look *inside* the archives the way you would read a table of contents. A TAR file stores a small header before each file, so we can list every member and its size without extracting 105 GB.

**Expected inputs**
- The four archives in `MyDrive/QPAD_Project/OULU_NPU/raw/`.

**Expected outputs**
- `QPAD_Project/manifests/archive_inventory.json`
- `QPAD_Project/manifests/archive_index_<archive>.json.gz` (member names, sizes and byte offsets; reused by every later notebook)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42
PEEK_MEMBERS = 200          # DRY_RUN: read only this many TAR headers per archive
if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb00", paths.logs / "nb00.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

## 1. Do the archives exist?
If a path is wrong, the error lists the nearby Drive folders (only the immediate parent folders — the Drive is never searched).

In [ ]:
from qpad.archive_io import check_archives_exist
ARCHIVES = {k: paths.archive(k) for k in ("train", "dev", "test", "metadata")}
check_archives_exist(ARCHIVES)
expected_gb = {"train": 37.75, "dev": 28.15, "test": 39.21, "metadata": 0.057}
for k, p in ARCHIVES.items():
    gb = p.stat().st_size / 1e9
    print(f"{k:9s} {gb:8.2f} GB   (expected ~{expected_gb[k]} GB)   {p}")
print(f"total     {sum(p.stat().st_size for p in ARCHIVES.values()) / 1e9:8.2f} GB")
print(f"local free space: {disk_free_gb(LOCAL_WORK_ROOT):.1f} GB  -> never extract all archives at once")

## 2. Peek (dry run) or full header index
`DRY_RUN = True` reads only the first few TAR headers (seconds). `DRY_RUN = False` indexes every header of every archive once and caches the result on Drive (minutes; no payload is read).

In [ ]:
from qpad.archive_io import peek_tar, index_zip, summarize_members
disc = cfgs["dataset"]["discovery"]
EXTS = (disc["video_extensions"], disc["annotation_extensions"], disc["protocol_path_tokens"])
if DRY_RUN:
    peek = {k: (index_zip(p) if k == "metadata" else peek_tar(p, PEEK_MEMBERS)) for k, p in ARCHIVES.items()}
    for k, recs in peek.items():
        s = summarize_members(recs, *EXTS)
        print(f"\n== {k}: first {len(recs)} members ==")
        print("kinds:", s["kinds"]); print("extensions:", s["extensions"]); print("dirs:", s["directories"])
        for kind, names in s["sample_names"].items():
            print(f"  {kind}: {names[:4]}")
    INVENTORY = None
else:
    from qpad.pipeline import archive_inventory
    preflight("archive header index", 4, list(ARCHIVES.values()), [paths.manifests], LOCAL_WORK_ROOT,
              {"discovery": disc}, RANDOM_SEED, DRY_RUN, "reads headers only; minutes on Drive")
    INVENTORY = archive_inventory(paths, cfgs["dataset"], force=FORCE_REBUILD, logger=logger)

## 3. What did we discover?
Check the extensions, the directory layout, how videos pair with annotation files, and where the Protocol_2 lists live.

In [ ]:
import pandas as pd
if INVENTORY is not None:
    rows = []
    for k, a in INVENTORY["archives"].items():
        rows.append({"archive": k, "members": a["n_members"], "files": a["n_files"],
                     "videos": a["kinds"].get("video", 0), "annotations": a["kinds"].get("annotation", 0),
                     "protocol_files": a["kinds"].get("protocol", 0), "other": a["kinds"].get("other", 0),
                     "uncompressed_GB": round(a["total_uncompressed_bytes"] / 1e9, 2),
                     "videos_with_same_stem_annotation": a["stem_pairing"]["videos_with_same_stem_annotation"]})
    display(pd.DataFrame(rows))
    for k, a in INVENTORY["archives"].items():
        print(f"\n{k}: extensions {a['extensions']}")
        print(f"  top directories {list(a['directories'])[:5]}")
        for kind, names in a["sample_names"].items():
            print(f"  {kind}: {names[:3]}")
    print("\nProtocol files:", json.dumps(INVENTORY["protocol_files"], indent=1))
    print("Discovery:", json.dumps(INVENTORY["discovery"], indent=1))
else:
    print("Dry run: set DRY_RUN = False to build the full inventory.")

In [ ]:
# --- Final summary -----------------------------------------------------------------------------
print("Archives verified:", list(ARCHIVES))
if INVENTORY is not None:
    print("Inventory saved:", paths.manifests / "archive_inventory.json")
    print("Total uncompressed:", round(INVENTORY["discovery"]["total_uncompressed_bytes"] / 1e9, 2), "GB")

## Before you continue — inspect

- [ ] All four archives exist and sizes are close to the expected values.
- [ ] Video extension(s) and annotation extension(s) match `configs/dataset.yaml → discovery`.
- [ ] Every video has a same-stem annotation file (`videos_with_same_stem_annotation`).
- [ ] Protocol_2 list files were found (Train/Dev/Test). If not, the audit relies on the subject/PAI rule.
- [ ] Each TAR contains ~1,800/1,350/1,800 videos (all PAI codes); only 2,970 belong to Protocol II.
- [ ] If the layout differs from these assumptions, stop and update the config before notebook 01.

**Next notebook:** `01_dataset_audit.ipynb`